In [ ]:
!pip install scikit-learn
!pip install h5py

In [ ]:
# ============================================================================
# CMU-MOSEI Text-Based Sentiment Classification — Kaggle Benchmark Task
# ============================================================================
# Task: Given a sentence transcript from CMU-MOSEI YouTube monologues,
#        predict the sentiment. TEXT input ONLY — no audio or video.
# Input: Single sentence transcript
# Output: One of 3 sentiment labels (negative, neutral, positive)
# Metric: Weighted-Average F1 Score (float, 0.0 - 1.0)
# Samples: 250 (stratified across sentiment classes, deterministic)
# ============================================================================

import kaggle_benchmarks as kbench
from dataclasses import dataclass
from sklearn.metrics import f1_score, classification_report
import pandas as pd
import numpy as np
import h5py
import os
import re

# ============================================================================
# CONFIGURATION
# ============================================================================

DATASET_ROOT = "/kaggle/input/datasets/samarwarsi/cmu-mosei"
NUM_SAMPLES = 250
RANDOM_SEED = 42
MIN_WORDS = 3

VALID_SENTIMENTS = ["negative", "neutral", "positive"]
SENTIMENT_INDEX = 0

SENTIMENT_SYNONYMS = {
    "neg": "negative", "bad": "negative", "unhappy": "negative",
    "sad": "negative", "angry": "negative", "terrible": "negative",
    "awful": "negative", "horrible": "negative",
    "none": "neutral", "mixed": "neutral", "flat": "neutral",
    "neither": "neutral", "ambiguous": "neutral",
    "pos": "positive", "good": "positive", "happy": "positive",
    "great": "positive", "excellent": "positive", "wonderful": "positive",
}


# ============================================================================
# STRUCTURED OUTPUT SCHEMA
# ============================================================================

@dataclass
class SentimentPrediction:
    """The LLM must return exactly one sentiment label."""
    sentiment: str


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def find_csd_files(dataset_root):
    """Locate the TimestampedWords.csd and Labels.csd files."""
    words_path = None
    labels_path = None

    for dirpath, dirnames, filenames in os.walk(dataset_root):
        for fname in filenames:
            fl = fname.lower()
            full = os.path.join(dirpath, fname)
            if "timestampedwords" in fl and fl.endswith(".csd"):
                words_path = full
            if "labels" in fl and fl.endswith(".csd"):
                labels_path = full

    if words_path is None:
        raise FileNotFoundError(
            f"Could not find CMU_MOSEI_TimestampedWords.csd under {dataset_root}"
        )
    if labels_path is None:
        raise FileNotFoundError(
            f"Could not find CMU_MOSEI_Labels.csd under {dataset_root}"
        )

    print(f"[INFO] Words CSD:  {words_path}")
    print(f"[INFO] Labels CSD: {labels_path}")
    return words_path, labels_path


def load_csd_file(csd_path):
    """Load a .csd file (HDF5 format) and return the data dictionary."""
    data = {}
    with h5py.File(csd_path, "r") as f:
        root_name = None
        for key in f.keys():
            if "data" in f[key]:
                root_name = key
                break

        if root_name is None:
            if "data" in f:
                data_group = f["data"]
            else:
                for key in f.keys():
                    if isinstance(f[key], h5py.Group):
                        root_name = key
                        break
                if root_name:
                    data_group = f[root_name]
                    if "data" in data_group:
                        data_group = data_group["data"]
                else:
                    raise ValueError(
                        f"Cannot parse CSD structure in {csd_path}. Keys: {list(f.keys())}"
                    )
        else:
            data_group = f[root_name]["data"]

        for seg_key in data_group.keys():
            seg = data_group[seg_key]
            entry = {}
            if "features" in seg:
                entry["features"] = seg["features"][()]
            if "intervals" in seg:
                entry["intervals"] = seg["intervals"][()]
            data[seg_key] = entry

    return data


def extract_sentences(words_data):
    """Reconstruct sentences from timestamped word data."""
    sentences = {}
    for seg_key, entry in words_data.items():
        features = entry.get("features")
        if features is None:
            continue

        words = []
        for item in features:
            if isinstance(item, (bytes, np.bytes_)):
                word = item.decode("utf-8", errors="ignore").strip()
            elif isinstance(item, np.ndarray):
                val = item.flat[0] if item.size > 0 else b""
                if isinstance(val, (bytes, np.bytes_)):
                    word = val.decode("utf-8", errors="ignore").strip()
                else:
                    word = str(val).strip()
            else:
                word = str(item).strip()

            if word and word not in ("", "sp", "SP", "<s>", "</s>"):
                words.append(word)

        sentence = " ".join(words)
        if len(words) >= MIN_WORDS:
            sentences[seg_key] = sentence

    return sentences


def extract_sentiment_labels(labels_data):
    """Extract sentiment scores from labels data."""
    sentiments = {}
    for seg_key, entry in labels_data.items():
        features = entry.get("features")
        if features is None:
            continue

        if features.ndim == 2:
            score = float(features[0, SENTIMENT_INDEX])
        elif features.ndim == 1:
            score = float(features[SENTIMENT_INDEX])
        else:
            continue

        if not np.isnan(score):
            sentiments[seg_key] = score

    return sentiments


def discretize_sentiment(score):
    """Convert continuous [-3, +3] score to 3-class label."""
    if score < 0:
        return "negative"
    elif score > 0:
        return "positive"
    else:
        return "neutral"


def build_dataset(sentences, sentiments):
    """Join sentences with sentiment labels by segment key."""
    rows = []
    common_keys = set(sentences.keys()) & set(sentiments.keys())
    print(f"[INFO] Segments with both text and labels: {len(common_keys)}")

    for key in sorted(common_keys):
        score = sentiments[key]
        rows.append({
            "segment_key": key,
            "sentence": sentences[key],
            "sentiment_score": score,
            "sentiment_class": discretize_sentiment(score),
            "word_count": len(sentences[key].split()),
        })

    df = pd.DataFrame(rows)
    print(f"[INFO] Built dataset with {len(df)} valid samples.")
    print(f"[INFO] Sentiment distribution:")
    print(df["sentiment_class"].value_counts().to_string())
    return df


def select_stratified_samples(df, n=NUM_SAMPLES):
    """Select N samples via stratified sampling across sentiment classes."""
    if len(df) == 0:
        raise ValueError("No valid samples found.")

    actual_n = min(n, len(df))
    present_classes = [c for c in VALID_SENTIMENTS if c in df["sentiment_class"].values]

    if actual_n < len(present_classes):
        class_counts = df["sentiment_class"].value_counts()
        priority_classes = class_counts.index.tolist()[:actual_n]
        parts = []
        for cls in priority_classes:
            pool = df[df["sentiment_class"] == cls]
            parts.append(pool.sample(n=1, random_state=RANDOM_SEED))
        selected_df = pd.concat(parts)
    else:
        guaranteed_parts = []
        remaining = actual_n

        for cls in present_classes:
            pool = df[df["sentiment_class"] == cls]
            sampled = pool.sample(n=1, random_state=RANDOM_SEED)
            guaranteed_parts.append(sampled)
            remaining -= 1

        guaranteed_df = pd.concat(guaranteed_parts)
        guaranteed_idx = set(guaranteed_df.index)

        if remaining > 0:
            leftover = df[~df.index.isin(guaranteed_idx)]
            take = min(remaining, len(leftover))
            if take > 0:
                extra = leftover.sample(n=take, random_state=RANDOM_SEED)
                selected_df = pd.concat([guaranteed_df, extra])
            else:
                selected_df = guaranteed_df
        else:
            selected_df = guaranteed_df

    selected_df = selected_df.sort_values("segment_key").reset_index(drop=True)

    print(f"\n[INFO] Final selection: {len(selected_df)} samples")
    print(selected_df["sentiment_class"].value_counts().to_string())
    return selected_df


def normalize_sentiment(raw_prediction):
    """Normalize the LLM's predicted sentiment to a canonical label."""
    cleaned = raw_prediction.strip().lower()
    cleaned = re.sub(r"[^a-z]", "", cleaned)

    if cleaned in VALID_SENTIMENTS:
        return cleaned

    if cleaned in SENTIMENT_SYNONYMS:
        return SENTIMENT_SYNONYMS[cleaned]

    for sent in VALID_SENTIMENTS:
        if sent in cleaned:
            return sent

    return "INVALID"


def build_prompt(sentence):
    """Build the classification prompt with the sentence."""
    return (
        "You are an expert sentiment analyst.\n\n"
        "Read the following sentence spoken by a person in a YouTube video "
        "and classify its overall sentiment.\n\n"
        f'Sentence: "{sentence}"\n\n'
        "Classify the sentiment as exactly one of:\n"
        "negative, neutral, positive\n\n"
        "Respond with a single JSON object containing one field 'sentiment' "
        "with your chosen label."
    )


# ============================================================================
# MAIN BENCHMARK TASK
# ============================================================================

@kbench.task(
    name="cmu_mosei_text_sentiment",
    description=(
        "Text-based sentiment classification on the CMU-MOSEI dataset. "
        "The model receives a single sentence transcript from a YouTube "
        "monologue and must classify the sentiment as one of 3 categories: "
        "negative, neutral, positive."
    ),
)
def cmu_mosei_text_sentiment(llm) -> float:
    """
    Evaluate an LLM's ability to classify sentiment from text only.
    Returns Weighted-Average F1 score between 0.0 and 1.0.
    """
    # STEP 1: Locate CSD files
    words_path, labels_path = find_csd_files(DATASET_ROOT)

    # STEP 2: Load CSD files (HDF5 format)
    print("[INFO] Loading words data...")
    words_data = load_csd_file(words_path)
    print(f"[INFO] Words data: {len(words_data)} segments loaded.")

    print("[INFO] Loading labels data...")
    labels_data = load_csd_file(labels_path)
    print(f"[INFO] Labels data: {len(labels_data)} segments loaded.")

    # STEP 3: Extract sentences and sentiment labels
    print("[INFO] Extracting sentences from word data...")
    sentences = extract_sentences(words_data)
    print(f"[INFO] Extracted {len(sentences)} valid sentences (>={MIN_WORDS} words).")

    print("[INFO] Extracting sentiment labels...")
    sentiments = extract_sentiment_labels(labels_data)
    print(f"[INFO] Extracted {len(sentiments)} sentiment scores.")

    # STEP 4: Build aligned dataset
    df = build_dataset(sentences, sentiments)

    # STEP 5: Select stratified samples
    samples = select_stratified_samples(df, n=NUM_SAMPLES)

    # STEP 6: Run predictions
    y_true = []
    y_pred = []
    results_log = []
    total = len(samples)

    for i, (idx, row) in enumerate(samples.iterrows()):
        ground_truth = row["sentiment_class"]
        sentence = row["sentence"]
        score = row["sentiment_score"]
        seg_key = row["segment_key"]

        display_text = sentence[:80] + "..." if len(sentence) > 80 else sentence
        print(f"\n[SAMPLE {i+1}/{total}] {seg_key}")
        print(f"  Text: \"{display_text}\"")
        print(f"  Score: {score:.2f} -> Ground truth: {ground_truth}")

        prompt_text = build_prompt(sentence)

        with kbench.chats.new(f"sample_{i}"):
            try:
                prediction = llm.prompt(
                    prompt_text,
                    schema=SentimentPrediction,
                )
                raw_sentiment = prediction.sentiment

            except Exception as e_schema:
                print(f"  [WARN] Schema failed ({type(e_schema).__name__}): {e_schema}")
                print(f"  [FALLBACK] Trying without schema...")

                with kbench.chats.new(f"sample_{i}_retry"):
                    raw_response = llm.prompt(prompt_text)
                    raw_sentiment = raw_response.strip()

        predicted = normalize_sentiment(raw_sentiment)

        print(f"  Raw: '{raw_sentiment}' -> Normalized: '{predicted}'")
        print(f"  {'CORRECT' if predicted == ground_truth else 'WRONG'}")

        y_true.append(ground_truth)
        y_pred.append(predicted)
        results_log.append({
            "sample": i + 1,
            "segment_key": seg_key,
            "sentence": display_text,
            "score": score,
            "ground_truth": ground_truth,
            "predicted": predicted,
            "correct": predicted == ground_truth,
        })

    # STEP 7: Compute Weighted-Average F1 Score
    all_labels = VALID_SENTIMENTS + (["INVALID"] if "INVALID" in y_pred else [])

    waf1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        labels=all_labels,
        zero_division=0,
    )

    # STEP 8: Print summary
    print("\n" + "=" * 70)
    print("RESULTS SUMMARY")
    print("=" * 70)

    results_df = pd.DataFrame(results_log)
    print(results_df[["sample", "ground_truth", "predicted", "correct"]].to_string(
        index=False
    ))

    correct_count = sum(r["correct"] for r in results_log)
    invalid_count = sum(1 for p in y_pred if p == "INVALID")

    print(f"\nAccuracy:  {correct_count}/{total} ({correct_count/total:.1%})")
    print(f"Invalid:   {invalid_count}/{total}")
    print(f"WAF1:      {waf1:.4f}")

    if total >= 3:
        print(f"\nPer-class classification report:")
        print(classification_report(
            y_true, y_pred, labels=VALID_SENTIMENTS, zero_division=0
        ))

    print("=" * 70)

    # STEP 9: Assertions — required for Kaggle to recognize the task
    kbench.assertions.assert_true(
        0.0 <= waf1 <= 1.0,
        expectation="Weighted-Average F1 score should be between 0 and 1."
    )

    kbench.assertions.assert_true(
        invalid_count < total,
        expectation="At least one prediction should be a valid sentiment label."
    )

    kbench.assertions.assert_true(
        correct_count > 0,
        expectation="Model should correctly classify at least one sample."
    )

    return waf1


# ============================================================================
# RUN THE TASK
# ============================================================================

cmu_mosei_text_sentiment.run(kbench.llm)

In [ ]:
%choose cmu_mosei_text_sentiment